In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install rdkit xgboost

import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdFingerprintGenerator
RDLogger.DisableLog("rdApp.*")   # hides harmless RDKit warnings

clean = pd.read_csv("/content/drive/MyDrive/NIR_Thesis/Data/my_data_clean.csv")   # or your Google Drive path
print(clean.shape)
print(clean.split.value_counts())

(17850, 9)
split
train    14281
val       1790
test      1779
Name: count, dtype: int64


In [3]:
morgan = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)

mol = Chem.MolFromSmiles(clean.chrom.iloc[100])
fp = morgan.GetCountFingerprintAsNumPy(mol)
print(len(fp))
print("non-zero positions:", (fp > 0).sum())
print("largest count:", fp.max())

1024
non-zero positions: 23
largest count: 9


In [4]:
desc = Descriptors.CalcMolDescriptors(mol)
print(len(desc))
pd.Series(desc).head(20)

217


,0
MaxAbsEStateIndex,5.598107
MaxEStateIndex,5.598107
MinAbsEStateIndex,0.972920
MinEStateIndex,0.972920
qed,0.304262
SPS,11.277778
MolWt,226.278000
HeavyAtomMolWt,216.198000
ExactMolWt,226.078250
NumValenceElectrons,82.000000


In [5]:
def conjugation_features(m):
    n_bonds = max(m.GetNumBonds(), 1)
    n_atoms = max(m.GetNumHeavyAtoms(), 1)
    conj_bonds = sum(b.GetIsConjugated() for b in m.GetBonds())
    sp2_sp = sum(a.GetHybridization() in (Chem.HybridizationType.SP2, Chem.HybridizationType.SP)
                 for a in m.GetAtoms())
    aromatic = sum(a.GetIsAromatic() for a in m.GetAtoms())
    return [conj_bonds,
            conj_bonds / n_bonds,
            sp2_sp / n_atoms,
            aromatic / n_atoms,
            Chem.rdMolDescriptors.CalcNumAromaticRings(m)]

conjugation_features(mol)

[21, 1.0, 1.0, 0.8888888888888888, 4]

In [6]:
def chrom_features(smi):
    m = Chem.MolFromSmiles(smi)
    fp = morgan.GetCountFingerprintAsNumPy(m)              # 1024
    desc = list(Descriptors.CalcMolDescriptors(m).values()) # 217
    conj = conjugation_features(m)                          # 5
    return np.concatenate([fp, desc, conj]).astype(float)

solv_morgan = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=256)

def solv_features(smi):
    m = Chem.MolFromSmiles(smi)
    fp = solv_morgan.GetCountFingerprintAsNumPy(m)         # 256
    desc = [Descriptors.MolLogP(m), Descriptors.TPSA(m), Descriptors.MolWt(m),
            Descriptors.NumHDonors(m), Descriptors.NumHAcceptors(m),
            Chem.rdMolDescriptors.CalcNumAromaticRings(m)]  # 6
    return np.concatenate([fp, desc]).astype(float)

print(len(chrom_features(clean.chrom.iloc[0])), len(solv_features(clean.solv.iloc[0])))

1246 262


In [7]:
chrom_cache = {s: chrom_features(s) for s in clean.chrom.unique()}
solv_cache = {s: solv_features(s) for s in clean.solv.unique()}

X = np.array([np.concatenate([chrom_cache[c], solv_cache[s]])
              for c, s in zip(clean.chrom, clean.solv)])
print(X.shape)
print("NaN values:", np.isnan(X).sum())

(17850, 1508)
NaN values: 648


In [8]:
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
X = np.clip(X, -1e6, 1e6)   # a few descriptors can blow up to huge values
print("NaN values after cleaning:", np.isnan(X).sum())

NaN values after cleaning: 0


In [9]:
import xgboost as xgb

y = clean.emi_nm.values
train = (clean.split == "train").values
val = (clean.split == "val").values
test = (clean.split == "test").values

model = xgb.XGBRegressor(
    n_estimators=3000,        # maximum number of trees
    learning_rate=0.03,       # how much each new tree corrects the previous ones
    max_depth=8,              # how complex each tree can be
    subsample=0.8,            # each tree sees a random 80% of rows
    colsample_bytree=0.4,     # ...and a random 40% of features
    min_child_weight=2,
    tree_method="hist",
    early_stopping_rounds=150,
    random_state=42,
    n_jobs=-1,
)
model.fit(X[train], y[train], eval_set=[(X[val], y[val])], verbose=500)
print("best number of trees:", model.best_iteration)

[0]	validation_0-rmse:95.07011
[500]	validation_0-rmse:51.41641
[1000]	validation_0-rmse:50.83357
[1180]	validation_0-rmse:50.79632
best number of trees: 1030


In [10]:
from sklearn.metrics import mean_absolute_error, r2_score

pred = model.predict(X[test])
true = y[test]
print(f"Test MAE: {mean_absolute_error(true, pred):.1f} nm   R²: {r2_score(true, pred):.3f}")

for lo, hi in [(0, 500), (500, 650), (650, 750), (750, 2000)]:
    m = (true >= lo) & (true < hi)
    print(f"{lo}-{hi} nm  n={m.sum():4d}  MAE={mean_absolute_error(true[m], pred[m]):5.1f}  "
          f"bias={np.mean(pred[m] - true[m]):6.1f}")

Test MAE: 33.6 nm   R²: 0.731
0-500 nm  n= 952  MAE= 28.0  bias=  14.7
500-650 nm  n= 684  MAE= 33.7  bias= -14.9
650-750 nm  n= 100  MAE= 60.7  bias= -49.6
750-2000 nm  n=  43  MAE= 93.1  bias= -92.1


In [11]:
train_pred = model.predict(X[train])
print(f"Train MAE: {mean_absolute_error(y[train], train_pred):.1f} nm")

Train MAE: 8.1 nm


In [12]:
import os
from sklearn.metrics import mean_squared_error

out = "/content/drive/MyDrive/NIR_Thesis/results"
os.makedirs(out, exist_ok=True)   # creates the results folder if it doesn't exist

# 1. Per-molecule test predictions (needed to compare with Chemprop later)
test_df = clean[test][["chrom", "solv", "emi_nm"]].copy()
test_df["xgb_pred_nm"] = pred
test_df.to_csv(f"{out}/xgb_test_predictions.csv", index=False)

# 2. Summary results table
rows = [{"bin": "overall", "n": len(true),
         "MAE": mean_absolute_error(true, pred),
         "RMSE": mean_squared_error(true, pred) ** 0.5,
         "R2": r2_score(true, pred),
         "bias": np.mean(pred - true)}]
for lo, hi in [(0, 500), (500, 650), (650, 750), (750, 2000)]:
    m = (true >= lo) & (true < hi)
    rows.append({"bin": f"{lo}-{hi}", "n": int(m.sum()),
                 "MAE": mean_absolute_error(true[m], pred[m]),
                 "bias": np.mean(pred[m] - true[m])})
pd.DataFrame(rows).round(3).to_csv(f"{out}/xgb_results.csv", index=False)

# 3. The trained model itself
model.save_model(f"{out}/xgb_model.json")

print(os.listdir(out))

['xgb_test_predictions.csv', 'xgb_results.csv', 'xgb_model.json']
